# Prediksi Ulasan Buruk Pelanggan & Analisis Fitur Teks (NLP)
**World Model for Business (Kasus Marketplace Olist)**  

---
Tujuan: Memprediksi risiko timbulnya ulasan buruk (bintang 1 atau 2) di sepanjang siklus hidup pesanan: **Titik T0** (checkout), **Titik T1** (penyerahan ke kurir), dan **Titik T2** (kedatangan barang), serta menguji secara empiris apakah integrasi fitur pemrosesan bahasa alami (NLP) dari ulasan historis seller memberikan nilai tambah prediktif yang sepadan dengan kompleksitas komputasinya.

In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

from src.models.classification import build_classification_pipeline, evaluate_top_k
from src.models.nlp import get_past_seller_reviews, extract_topics

sns.set_theme(style='whitegrid', palette='muted')

df_t0 = pd.read_parquet('../data/processed/features_t0.parquet')
df_t1 = pd.read_parquet('../data/processed/features_t1.parquet')
df_t2 = pd.read_parquet('../data/processed/features_t2.parquet')
df_raw = pd.read_parquet('../data/processed/dev_orders.parquet')

# Bersihkan baris dengan target null
df_t0 = df_t0.dropna(subset=['is_bad_review'])
df_t1 = df_t1.dropna(subset=['is_bad_review'])
df_t2 = df_t2.dropna(subset=['is_bad_review'])

## 1. Pemodelan Bertahap Sepanjang Siklus Transaksi (Titik T0, T1, dan T2)
Kita mengevaluasi bagaimana penambahan informasi operasional secara bertahap (dari saat pemesanan, pengemasan seller, hingga durasi transit kurir) meningkatkan ketajaman prediksi ketidakpuasan pelanggan.

In [2]:
def time_split(df):
    train = df[df['order_purchase_timestamp'] < '2018-04-01']
    test = df[df['order_purchase_timestamp'] >= '2018-04-01']
    return train, test

cat_cols = ['primary_category']
num_cols_t0 = ['total_items_value', 'total_freight_value', 'estimated_delivery_days', 'avg_product_weight_g', 'seller_recent_avg_score_30d']
num_cols_t1 = num_cols_t0 + ['seller_handling_days']
num_cols_t2 = num_cols_t1 + ['actual_delivery_days']

results = []

for name, df_curr, num_cols in [('T0 (Checkout)', df_t0, num_cols_t0), ('T1 (Kurir)', df_t1, num_cols_t1), ('T2 (Delivered)', df_t2, num_cols_t2)]:
    train, test = time_split(df_curr)
    X_train, y_train = train[cat_cols + num_cols], train['is_bad_review']
    X_test, y_test = test[cat_cols + num_cols], test['is_bad_review']
    
    model = build_classification_pipeline(cat_cols, num_cols, 'logistic')
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    
    p, r, auc_val = evaluate_top_k(y_test, probs, k_fraction=0.10)
    results.append({'Titik Evaluasi': name, 'Top-10% Precision': f'{p*100:.1f}%', 'Top-10% Recall': f'{r*100:.1f}%', 'PR-AUC': round(auc_val, 3)})

res_df = pd.DataFrame(results)
display(res_df)

,Titik Evaluasi,Top-10% Precision,Top-10% Recall,PR-AUC
0,T0 (Checkout),21.0%,17.7%,0.175
1,T1 (Kurir),22.3%,19.6%,0.186
2,T2 (Delivered),30.7%,28.3%,0.259


## 2. Eksperimen Nilai Tambah Fitur Teks Ulasan Historis (NLP / TF-IDF)
Kita menguji apakah riwayat ulasan tekstual seller masa lalu yang diolah dengan TF-IDF mampu meningkatkan PR-AUC model pada titik T0 dengan ambang signifikansi minimum kenaikan $\Delta \ge 0,02$.

In [3]:
print("Mengekstrak teks masa lalu (butuh beberapa detik)...")

# Buat kamus ulasan seller
review_dict = {}
for _, row in df_raw.dropna(subset=['review_comment_message', 'order_purchase_timestamp']).iterrows():
    sid = row['primary_seller_id']
    if sid not in review_dict: review_dict[sid] = []
    review_dict[sid].append((row['order_purchase_timestamp'], row['review_comment_message']))

train_t0, test_t0 = time_split(df_t0)

# Ambil sampel representatif untuk komparasi efisien
train_t0_sample = train_t0.head(10000)
test_t0_sample = test_t0.head(5000)

def add_text_feature(df_in):
    texts = []
    for _, row in df_in.iterrows():
        t = get_past_seller_reviews(df_raw, row['order_purchase_timestamp'], row.get('primary_seller_id', ''), review_dict)
        texts.append(t)
    return texts

text_train = [t if len(t)>2 else 'no_review' for t in add_text_feature(train_t0_sample)]
text_test = [t if len(t)>2 else 'no_review' for t in add_text_feature(test_t0_sample)]

tfidf = TfidfVectorizer(max_features=100)
X_train_text = tfidf.fit_transform(text_train)
X_test_text = tfidf.transform(text_test)

# Model Tabular Murni (Tanpa Teks)
model_no_text = build_classification_pipeline(cat_cols, num_cols_t0, 'logistic')
X_train_num, y_train = train_t0_sample[cat_cols + num_cols_t0], train_t0_sample['is_bad_review']
X_test_num, y_test = test_t0_sample[cat_cols + num_cols_t0], test_t0_sample['is_bad_review']

model_no_text.fit(X_train_num, y_train)
probs_no = model_no_text.predict_proba(X_test_num)[:, 1]
_, _, auc_no = evaluate_top_k(y_test, probs_no, 0.1)

# Model Gabungan Tabular + Teks NLP
X_train_processed = model_no_text.named_steps['preprocessor'].transform(X_train_num)
X_test_processed = model_no_text.named_steps['preprocessor'].transform(X_test_num)

X_train_comb = hstack([X_train_processed, X_train_text])
X_test_comb = hstack([X_test_processed, X_test_text])

from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(class_weight='balanced', max_iter=200)
clf.fit(X_train_comb, y_train)
probs_yes = clf.predict_proba(X_test_comb)[:, 1]
_, _, auc_yes = evaluate_top_k(y_test, probs_yes, 0.1)

print(f"PR-AUC Tanpa Teks : {auc_no:.4f}")
print(f"PR-AUC Dengan NLP : {auc_yes:.4f}")
print(f"Selisih Kenaikan  : {auc_yes - auc_no:.4f}")

if (auc_yes - auc_no) < 0.02:
    print("\nKEPUTUSAN ARSITEKTUR: Kenaikan < 0.02. Fitur teks tidak memberikan nilai tambah signifikan.")
else:
    print("\nKEPUTUSAN ARSITEKTUR: NLP memberikan kenaikan signifikan.")

Mengekstrak teks masa lalu (butuh beberapa detik)...


PR-AUC Tanpa Teks : 0.1700
PR-AUC Dengan NLP : 0.1700
Selisih Kenaikan  : 0.0000

KEPUTUSAN ARSITEKTUR: Kenaikan < 0.02. Fitur teks tidak memberikan nilai tambah signifikan.


---
## 3. Ringkasan Eksekutif & Keputusan Arsitektur Fitur

1. **Dinamika Informasi Sepanjang Siklus Pesanan**:
   - Di titik **T0 (Saat Checkout)**: Top-10% Precision = **21,0%**, PR-AUC = **0,175**.
   - Di titik **T1 (Penyerahan ke Kurir)**: Top-10% Precision = **22,3%**, PR-AUC = **0,186**.
   - Di titik **T2 (Kedatangan Paket / Delivered)**: Top-10% Precision melonjak drastis ke **30,7%**, dengan PR-AUC **0,259**. Hal ini mengonfirmasi secara empiris bahwa kepastian durasi aktual kedatangan pengiriman adalah prediktor terkuat kepuasan pelanggan.

2. **Hasil Eksperimen Fitur Teks (NLP)**:
   - PR-AUC Tanpa Teks: **0,1700** vs Dengan NLP: **0,1700** (Kenaikan: **0,0000**).
   - Penambahan fitur TF-IDF dari riwayat komentar ulasan seller tidak memberikan peningkatan daya prediksi baru dibanding fitur tabular agregat (`seller_recent_avg_score_30d`).
   - **Keputusan Rekayasa Fitur**: Fitur teks dieliminasi dari sistem produksi untuk menjaga kecepatan inferensi dan efisiensi memori, dengan performa prediktif yang tetap optimal.